In [1]:
import spacy
from fastcoref import spacy_component
import requests

PORT = 8008
SERVER = "http://127.0.0.1"

podcasts = requests.get(f"{SERVER}:{PORT}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']}")

/home/adamj/anaconda3/envs/whspr/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0: Verdict with Ted Cruz
1: Ta Kommandoen med Geir Aker
2: Misjonen med Antonsen og Golden
3: Leger om livet
4: Norsken, svensken og dansken
5: Burde vært pensum
6: Huberman Lab
7: Stuff You Should Know
8: The Ben Shapiro Show
9: The Megyn Kelly Show
10: Pivot
11: The Daily
12: The Ezra Klein Show
13: Lex Fridman Podcast
14: Checks and Balance from The Economist
15: Money Talks from The Economist
16: The Economist Asks
17: The Ramsey Show
18: Dateline NBC
19: Pod Save America
20: Freakonomics Radio


In [2]:
import spacy
from fastcoref import spacy_component
import requests

for podcast in podcasts:
    if not podcast["language"].startswith("en"):
        print(f"skipping {podcast['title']}, language is {podcast['language']}")
        continue
    for audioitem in podcast['audioitem_set']:
        #if audioitem['title'] != "Ep. 1707 - World Famous YouTuber MrBeast Hit With Trans Controversy":
        #    continue
        print("starting episode: ", audioitem['title'], podcast['title'])
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == "spaCy":
                    seg_uuid = segmentation['uuid']
                    seg = requests.get(f"{SERVER}:{PORT}/api/segmentations/{seg_uuid}/")
                    seg = seg.json()
                    utterances = seg["utterance_set"]
                    # check if coref already exists in any of this segmentation's utterances
                    if any([utt["text_coref"] for utt in utterances]):
                        print("skipping ", audioitem['title'], podcast['title'])
                        continue
                    nlp = spacy.load("en_core_web_lg")
                    nlp.add_pipe(
                        "fastcoref", 
                        config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
                    )

                    # add a context field to each utterance with the 50 previous utterances
                    for i, utt in enumerate(utterances):
                        utt["context"] = " ".join([u["text"] for u in utterances[max(0, i-50):i+1]])

                    docs = nlp.pipe(
                    [utterance["context"] for utterance in utterances],
                    component_cfg={"fastcoref": {'resolve_text': True}}
                    )
                    try:
                        docs = list(docs)
                    except:
                        print("error with ", audioitem['title'], podcast['title'])
                        for doc in docs:
                            print(doc)
                        continue

                    assert len(docs) == len(utterances)

                    for i, doc in enumerate(docs):
                        resolved_text = doc._.resolved_text

                        # Create a new Doc object without running the entire pipeline
                        sentences_doc = nlp.make_doc(resolved_text)

                        # Apply the "senter" component to the sentences_doc
                        nlp.get_pipe("senter")(sentences_doc)

                        first = coref = next(sentences_doc.sents)
                        for coref in sentences_doc.sents: pass

                        first = original = next(doc.sents)
                        for original in doc.sents: pass
                        if coref.text.strip() != original.text.strip():
                            # write to API
                            utt_uuid = utterances[i]["uuid"]
                            utterances[i]["text_coref"] = coref.text.strip()
                            # no changes to these child record, so remove them
                            utterances[i].pop("classification_set")
                            utterances[i].pop("query_set")
                            res = requests.post(f"{SERVER}:{PORT}/api/utterances/{utt_uuid}/", json=utterances[i])
                            print(res.status_code)

                    

starting episode:  Trump Pleads Not Guilty - What Happens Now? Verdict with Ted Cruz
skipping  Trump Pleads Not Guilty - What Happens Now? Verdict with Ted Cruz
starting episode:  Trump Indictment - What Are Trump's Options Legally, To Fight Back! Verdict with Ted Cruz
skipping  Trump Indictment - What Are Trump's Options Legally, To Fight Back! Verdict with Ted Cruz
starting episode:  Trump Indictment - What It Means & What's Next Verdict with Ted Cruz
skipping  Trump Indictment - What It Means & What's Next Verdict with Ted Cruz
starting episode:  How Can We STOP School Shootings - and, Maddeningly, Why Democrats Keep Blocking Real School Safety Legislation Verdict with Ted Cruz
skipping  How Can We STOP School Shootings - and, Maddeningly, Why Democrats Keep Blocking Real School Safety Legislation Verdict with Ted Cruz
starting episode:  Mayorkas LIED: Explosive Cross-Examination In The Senate Judiciary Committee Verdict with Ted Cruz
skipping  Mayorkas LIED: Explosive Cross-Examina

Some weights of the model checkpoint at biu-nlp/lingmess-coref were not used when initializing LingMessModel: ['longformer.embeddings.position_ids']
- This IS expected if you are initializing LingMessModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing LingMessModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
04/28/2023 18:14:38 - INFO - 	 missing_keys: []
04/28/2023 18:14:38 - INFO - 	 unexpected_keys: []
04/28/2023 18:14:38 - INFO - 	 mismatched_keys: []
04/28/2023 18:14:38 - INFO - 	 error_msgs: []
04/28/2023 18:14:38 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/28/2023 18:14:46 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:14:50 - INFO - 	 ***** Runnin

error with  Selects: Was There A Real Robin Hood? Stuff You Should Know
starting episode:  METI: Existential Threat? Probably Yes Stuff You Should Know
skipping  METI: Existential Threat? Probably Yes Stuff You Should Know
starting episode:  Why does everyone love Dolly Parton? Stuff You Should Know
skipping  Why does everyone love Dolly Parton? Stuff You Should Know
starting episode:  Should Rich Countries Cancel Poor Countries’ Debt? Stuff You Should Know
skipping  Should Rich Countries Cancel Poor Countries’ Debt? Stuff You Should Know
starting episode:  Short Stuff: Routines Stuff You Should Know
skipping  Short Stuff: Routines Stuff You Should Know
starting episode:  Short Stuff: Botox Brain Stuff You Should Know
skipping  Short Stuff: Botox Brain Stuff You Should Know
starting episode:  Carbon Monoxide: Please Just Listen Anyway Stuff You Should Know
skipping  Carbon Monoxide: Please Just Listen Anyway Stuff You Should Know
starting episode:  Ep. 1703 - Trump Is Charged -- And Th

04/28/2023 18:16:24 - INFO - 	 missing_keys: []
04/28/2023 18:16:24 - INFO - 	 unexpected_keys: []
04/28/2023 18:16:24 - INFO - 	 mismatched_keys: []
04/28/2023 18:16:24 - INFO - 	 error_msgs: []
04/28/2023 18:16:24 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/28/2023 18:16:31 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:16:34 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:14<00:00, 18.13it/s]
04/28/2023 18:16:59 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:17:04 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:21<00:00, 11.77it/s]
04/28/2023 18:17:37 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:17:41 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:21<00:00, 11.75it/s]


error with  Ep. 1714 - Corpse Declares Re-election Launch The Ben Shapiro Show
starting episode:  Ep. 1710 -  Want To Visit The White House? Don't Be White! The Ben Shapiro Show
skipping  Ep. 1710 -  Want To Visit The White House? Don't Be White! The Ben Shapiro Show
starting episode:  Ep. 1708 - The Real Interracial Crime Problem Isn't White On Black, It's Black On White The Ben Shapiro Show


...

error with  Why Are There So Many Bad Bosses? (Ep. 495 Replay) Freakonomics Radio
starting episode:  536. Is Your Plane Ticket Too Expensive — or Too Cheap? Freakonomics Radio
skipping  536. Is Your Plane Ticket Too Expensive — or Too Cheap? Freakonomics Radio
starting episode:  535. Why Is Flying Safer Than Driving? Freakonomics Radio
skipping  535. Why Is Flying Safer Than Driving? Freakonomics Radio
starting episode:  538. A Radically Simple Way to Boost a Neighborhood Freakonomics Radio
skipping  538. A Radically Simple Way to Boost a Neighborhood Freakonomics Radio
starting episode:  539. Why Does One Tiny State Set the Rules for Everyone? Freakonomics Radio


04/28/2023 18:42:30 - INFO - 	 missing_keys: []
04/28/2023 18:42:30 - INFO - 	 unexpected_keys: []
04/28/2023 18:42:30 - INFO - 	 mismatched_keys: []
04/28/2023 18:42:30 - INFO - 	 error_msgs: []
04/28/2023 18:42:30 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/28/2023 18:42:40 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:42:44 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:19<00:00, 13.47it/s]


error with  539. Why Does One Tiny State Set the Rules for Everyone? Freakonomics Radio
starting episode:  Why Your Projects Are Always Late — and What to Do About It (Ep. 323 Replay) Freakonomics Radio
skipping  Why Your Projects Are Always Late — and What to Do About It (Ep. 323 Replay) Freakonomics Radio
starting episode:  540. Swearing Is More Important Than You Think Freakonomics Radio


04/28/2023 18:43:08 - INFO - 	 missing_keys: []
04/28/2023 18:43:08 - INFO - 	 unexpected_keys: []
04/28/2023 18:43:08 - INFO - 	 mismatched_keys: []
04/28/2023 18:43:08 - INFO - 	 error_msgs: []
04/28/2023 18:43:08 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
04/28/2023 18:43:16 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:43:21 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:17<00:00, 14.96it/s]
04/28/2023 18:43:48 - INFO - 	 Tokenize 256 inputs...
04/28/2023 18:43:53 - INFO - 	 ***** Running Inference on 256 texts *****
Inference: 100%|██████████| 256/256 [00:20<00:00, 12.20it/s]

error with  540. Swearing Is More Important Than You Think Freakonomics Radio
